# 05. Audience Retention Modeling & Diagnostics
## Exponential Decay Model & Drop-off Detection
- Fit $R(t) = R_0 e^{-\lambda t} + C$
- Goodness of fit threshold ($R^2 \ge 0.70$)
- 0-30s Hook Drop-Off Rate
- Identifying Re-watch Spikes (rewind interest) and Dips (boredom / churn)


In [ ]:
import os
import sys
import sqlite3
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy.optimize import curve_fit

PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), ".."))
conn = sqlite3.connect(os.path.join(PROJECT_ROOT, "youtube_growth.db"))

# Fetch retention points for the top viral video
top_vid = pd.read_sql_query("SELECT video_id FROM derived_video_metrics ORDER BY virality_score DESC LIMIT 1", conn).iloc[0]['video_id']
df_ret = pd.read_sql_query(f"SELECT second, elapsed_ratio, viewer_percentage FROM audience_retention WHERE video_id = '{top_vid}' ORDER BY second ASC", conn)

def exp_decay(t, R0, decay_rate, C):
    return R0 * np.exp(-decay_rate * t) + C

t_data = df_ret['elapsed_ratio'].values
r_data = df_ret['viewer_percentage'].values

popt, _ = curve_fit(exp_decay, t_data, r_data, p0=[80, 1.5, 20], maxfev=5000)
preds = exp_decay(t_data, *popt)

ss_res = np.sum((r_data - preds) ** 2)
ss_tot = np.sum((r_data - np.mean(r_data)) ** 2)
r2 = 1.0 - (ss_res / max(1e-6, ss_tot))

print(f"Fitted Model: R(t) = {popt[0]:.2f} * e^(-{popt[1]:.2f} * t) + {popt[2]:.2f}")
print(f"Model Goodness of Fit R^2: {r2:.4f}")


In [ ]:
# Plot Fitted Decay vs Actual Retention
plt.figure(figsize=(9, 4.5))
plt.plot(df_ret['elapsed_ratio'] * 100, r_data, label='Actual Retention (%)', color='cyan', lw=2)
plt.plot(df_ret['elapsed_ratio'] * 100, preds, '--', label=f'Exponential Fit (R2={r2:.2f})', color='orange', lw=2)
plt.axvline(x=5.0, color='red', linestyle=':', label='Hook Boundary (30s)')
plt.title(f"Retention Curve Dynamics for Video: {top_vid}")
plt.xlabel("Elapsed Video Duration (%)")
plt.ylabel("Audience Retention (%)")
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()
